In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

# from utils import Vcom
from utils import get_comoving_distance_h, c, z_min, get_comoving_distance, cosmo
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

# from Ng8_func import N_g_8, N_g_8_theory, select_bottom20_in_random, select_top20_in_random
from Ng8_func import N_g_8_cross, N_g_8_theory_cross, select_bottom20_in_random, select_top20_in_random, return_box_rsd_pos

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
Mr_bins_x

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
# rp_bins = np.logspace(-0.8, 1.6, 13)
# rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2
Mr_bins = np.linspace(-22, -18, 21)
Mr_bins_x = (Mr_bins[:-1] + Mr_bins[1:])/2
# Mr_bins_x
DDP_Mr_limit = -20

from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
Mr_bins_x

In [ ]:
# Boundary limited
def Vcom(zmax, angarea = 6369):
    return (4 * np.pi / 3 * (get_comoving_distance(zmax) ** 3 - get_comoving_distance(0.02) ** 3)
            * cosmo.h ** 3 * angarea / 41253)

In [ ]:
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

plt.figure(figsize=(15, 3))
plt.scatter(data['ra'], data['dec'], s = 0.1)

In [ ]:
from scipy.optimize import fsolve

def f1(x):
    """Western limit: δ < -2.555556(α - 131°)"""
    return -2.555556 * (x - 131)

def f2(x):
    """Northern limit: δ < arcsin[...] - 更简洁的版本"""
    angle_rad = np.deg2rad(x - 95.9)
    
    numerator = 0.93232 * np.sin(angle_rad)
    denominator = np.sqrt(1 - (0.93232 * np.cos(angle_rad))**2)
    
    # 使用np.where处理分母为零的情况
    argument = np.where(denominator != 0, numerator / denominator, 0)
    argument = np.clip(argument, -1, 1)
    
    return np.rad2deg(np.arcsin(argument))

def f3(x):
    """Eastern limit: δ < 1.70909(α - 235°)"""
    return 1.70909 * (x - 235)

def equation1(x):
    return f1(x) - f2(x)

def equation2(x):
    return f3(x) - f2(x)
# 求解交点
# 提供多个初始猜测值以找到所有交点
initial_guesses1 = [115]
intersections1 = []
initial_guesses2 = [255]
intersections2 = []
for guess in initial_guesses1:
    root = fsolve(equation1, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections1):
        intersections1.append(root[0])
for guess in initial_guesses2:
    root = fsolve(equation2, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections2):
        intersections2.append(root[0])

print("交点 x 坐标:", intersections1)
print("交点坐标:")
for x in intersections1:
    y = f1(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

print("交点 x 坐标:", intersections2)
print("交点坐标:")
for x in intersections2:
    y = f3(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

In [ ]:
def boundary_limit(stat_ra, stat_dec, ):
    # section west
    ra_mask  = (115.1858 < stat_ra) & (stat_ra < 131)
    dec_mask = (f1(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    western_mask = ra_mask & dec_mask
    # section middle
    ra_mask  = (131 < stat_ra) & (stat_ra < 235)
    dec_mask = (0 < stat_dec) & (stat_dec < f2(stat_ra))
    middle_mask = ra_mask & dec_mask
    # section east
    ra_mask  = (235 < stat_ra) & (stat_ra < 257.7142)
    dec_mask = (f3(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    eastern_mask = ra_mask & dec_mask
    # total mask
    boundary_mask = western_mask | middle_mask | eastern_mask

    return boundary_mask

In [ ]:
plt.figure(figsize=(12, 6))

plt.scatter(intersections1, [f1(x) for x in intersections1], 
           color='red', s=50, zorder=5)#, label='intersections1'
plt.scatter(intersections2, [f3(x) for x in intersections2], 
           color='red', s=50, zorder=5)#, label='intersections2'
# 绘制函数进行比较
alpha_values = np.linspace(100, 300, 200)


# 绘制三个边界函数
plt.plot(alpha_values, f1(alpha_values), 'r-', label='Western: δ < -2.555556(α-131)', linewidth=2)
plt.plot(alpha_values, f2(alpha_values), 'b-', label='Northern: δ < arcsin[...]', linewidth=2)
plt.plot(alpha_values, f3(alpha_values), 'g-', label='Eastern: δ < 1.70909(α-235)', linewidth=2)

# 添加水平线 δ = 0 (southern limit)
plt.axhline(y=0, color='purple', linestyle='-', label='Southern: δ > 0°')

### SDSS plot 
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

boundary_mask = boundary_limit(data['ra'], data['dec'])

plt.scatter(data['ra'][boundary_mask], data['dec'][boundary_mask], s = 0.1)

# 设置图表属性
plt.xlabel(r'$\alpha (°)$')
plt.ylabel(r'$\delta (°)$')
plt.title('Boundary Limits Comparison')
plt.legend(fontsize = 14)
plt.grid(True, alpha=0.3)
plt.xlim(80, 300)
plt.ylim(-20, 90)

plt.show()

### $N_g^8$ percentile $w_p(r_p)$

In [ ]:
z_Mr = np.array([[-22, 73500],[-21.5, 59600], [-21, 47650], [-20.5, 39700], [-20, 31900], [-19.5, 25450], [-19, 19250], [-18.5, 15750],[-18,12500]])


In [ ]:
import numpy as np
from astropy.cosmology import Planck18, FlatLambdaCDM, z_at_value
from astropy import units as u

def redshift_from_magnitude(M_abs, m_app=17.6, cosmology=Planck13):
    """
    通过视星等和绝对星等计算红移。

    参数:
    m_app (float): 视星等
    M_abs (float): 绝对星等
    cosmology: 宇宙学模型，默认为Planck13

    返回:
    float: 红移 z
    """
    # 1. 计算距离模数
    distance_modulus = m_app - M_abs
    
    # 2. 计算光度距离 (单位：秒差距)
    d_L_pc = 10 ** ((distance_modulus + 5) / 5)
    
    # 3. 转换为Astropy单位（兆秒差距更常用）
    # d_L = d_L_pc * u.pc
    # 或者更常用 Mpc
    d_L_mpc = (d_L_pc / 1e6) * u.Mpc / cosmology.h
    
    # 4. 使用宇宙学模型求解红移z
    #    z_at_value 函数数值求解：cosmology.luminosity_distance(z) = d_L
    try:
        z = z_at_value(cosmology.luminosity_distance, d_L_mpc)
        return z
    except Exception as e:
        print(f"求解失败: {e}")
        return None

# 示例1：近距离天体（z ≈ 0）
plt.figure(figsize=(6, 6),dpi=100)

M1 = np.linspace(-18, -22, 21)
M1_x = (M1[1:] + M1[:-1])/2

z1 = np.zeros(len(M1_x))
for i in range(len(M1_x)):
    z1[i] = redshift_from_magnitude(M1_x[i])

z_thr = np.zeros(len(M1))
for i in range(len(M1)):
    z_thr[i] = redshift_from_magnitude(M1[i])

plt.plot(z1, M1_x, color='green')
plt.plot(z_Mr[:,1]/300000, z_Mr[:,0], color='gray')

plt.plot([0.02, 0.02], [-18, -22], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-20)], [-18, -18], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-20)], [-22, -22], lw = 2, c = 'black')
plt.plot([redshift_from_magnitude(-20), redshift_from_magnitude(-20)], [-18, -22], lw = 2, c = 'black', label = 'Ng8')

plt.plot([0.02, 0.02], [-20, -22], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-20)], [-20, -20], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-20)], [-22, -22], lw = 2, ls = '--', c = 'red')
plt.plot([redshift_from_magnitude(-20), redshift_from_magnitude(-20)], [-20, -22], lw = 2, ls = '--', c = 'red', label = 'DDP')

plt.plot([0.02, z_thr[0]], [M1[0], M1[0]], lw = 2, c = 'royalblue', label = 'env LF volume')

for i in range(10):
    plt.plot([0.02, z_thr[i]], [M1[i], M1[i]], lw = 2, c = 'royalblue')
    plt.plot([z_thr[i], z_thr[i]], [M1[i], M1[i+1]], lw = 2, c = 'royalblue')

plt.legend(fontsize = 16)

plt.xlabel('redshift')
plt.ylabel('Mr')
plt.xlim(0.01,0.25)
plt.gca().invert_yaxis()

#### check total LF, bright1 zmin = 0.02

In [ ]:
def Phi_LF(gal_Mr, gal_z, Mr_brighter, Mr_fainter):
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    ### intersection of z = 0.02 & z from MMIN = 14.5 ---- 2026/3/17
    z_mask  = ( 0.02 < gal_z) & (gal_z < z_max) 
    Phi_LF  = np.count_nonzero( Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    print(f'{np.count_nonzero(Mr_mask & z_mask)} | {Vcom(z_max):.2f} | {Phi_LF:.2e}')

    return Phi_LF

In [ ]:
def Phi_LF_theory(gal_Mr, Mr_brighter, Mr_fainter):
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    #############################################################################################
    Phi_LF  = np.count_nonzero( Mr_mask) / 1000**3 / Mr_bin_width
    print(f'{np.count_nonzero(Mr_mask )} | 1000^3 | {Phi_LF:.2e}')
    return Phi_LF

In [ ]:
def env_LF(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def env_LF_DDP(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter, DDP_Mr = DDP_Mr_limit):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    # z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    z_max   = redshift_from_magnitude(DDP_Mr) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = DDP_Mr_limit):
# def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins):

    env_LF_bottom20_red  = np.zeros(len(Mr_bins)-1)
    env_LF_bottom20_blue = np.zeros(len(Mr_bins)-1)
    env_LF_top20_red     = np.zeros(len(Mr_bins)-1)
    env_LF_top20_blue    = np.zeros(len(Mr_bins)-1)

    ### Ng8 ############################################################################################################
    # thr_mask    = gal_Mr < Mr_bins[i+1] # fainter end
    # # thr_mask    = (gal_Mr < Mr_bins[i+1]) & (gal_fac >= 0.9) # fainter end, f_ac completenesss (r = 8 Mpc/h) >= 0.9
    # z_max       = redshift_from_magnitude(Mr_bins[i+1]) # z_max depends on the the fainter Mr
    # z_mask      = (gal_z > z_min) & (gal_z < z_max)
    # # pos_mask    = (100 < gal_pos[:, 0]) & (gal_pos[:, 0] < 300)
    # mask        = thr_mask & z_mask #& pos_mask

    mask        = (-22 < gal_Mr) & (gal_Mr < -18) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))

    gal_pos_thr = gal_pos[mask]
    gal_Mr_thr  = gal_Mr[mask]
    gal_gmr_thr = gal_gmr[mask]
    gal_z_thr   = gal_z[mask]
    # gal_fac_thr = gal_fac[mask]

    gal_pos_DDP = gal_pos[(-22 < gal_Mr) & (gal_Mr < DDP_Mr) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))]

    Ng8_thr_DDP = N_g_8_cross(gal_pos_thr, gal_pos_DDP)
    # Ng8_thr    = N_g_8( D1_ra_thr, D1_dec_thr, D1_z_thr ) / gal_fac_thr

    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)
    D2_bot_Mr  = select_bottom20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_bot_gmr = select_bottom20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_bot_z   = select_bottom20_in_random(gal_z_thr  , Ng8_thr_DDP)

    # cross correlation with data and data(Ng8 top 20%)
    D2_top_Mr  = select_top20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_top_gmr = select_top20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_top_z   = select_top20_in_random(gal_z_thr  , Ng8_thr_DDP)
    ######################################################################################################################
    for i in range(len(Mr_bins)-1):
        if i < np.where(Mr_bins == DDP_Mr)[0][0] + 1: # Mr [-22, -19]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF_DDP(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF_DDP(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)

        else: # Mr [-19, -18]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1])
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1])
    ######################################################################################################################
    return env_LF_bottom20_red, env_LF_bottom20_blue, env_LF_top20_red, env_LF_top20_blue

#### Evironment dependent LF, Mr_C_bins

##### CCMD Main box 4Quads in Mr[-22,-21]

##### DDP_limit = -20

In [ ]:
### CCMD Mock01

paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

for path in paths:
    data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
    boundary_mask = boundary_limit(data['ra'], data['dec'])
    data = data[boundary_mask]
    
    gal_pos = np.array([np.array(data['ra']),
                        np.array(data['dec']),    
                        np.array(data['z'])]).T
    gal_Mr  = np.array(data['Mr'])
    gal_gmr = np.array(data['gmr'])
    gal_z   = np.array(data['z'])

    env_LF_bottom20_red, env_LF_bottom20_blue, \
    env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = -20)

    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
### CCMD 4 Quads cov
paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

bot_red_data  = []
bot_blue_data = []
top_red_data  = []
top_blue_data = []

for path in paths:
    bottom_red_temp  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_bottom20_red_env_LF.npy')
    bot_red_data.append(bottom_red_temp)
    bottom_blue_temp = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_bottom20_blue_env_LF.npy')
    bot_blue_data.append(bottom_blue_temp)  

    top_red_temp     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_top20_red_env_LF.npy')
    top_red_data.append(top_red_temp)
    top_blue_temp    = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit20_Mr_C_Ng8_top20_blue_env_LF.npy')
    top_blue_data.append(top_blue_temp)

bot_red_data  = np.array(bot_red_data)
bot_blue_data = np.array(bot_blue_data)
top_red_data  = np.array(top_red_data)
top_blue_data = np.array(top_blue_data)

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' , np.mean(bot_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy', np.mean(bot_blue_data, axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_top20_red_env_LF_mean.npy'    , np.mean(top_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   , np.mean(top_blue_data, axis=0))

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' , np.cov(bot_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy', np.cov(bot_blue_data, rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_top20_red_env_LF_cov.npy'    , np.cov(top_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit20_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   , np.cov(top_blue_data, rowvar=False))

##### DDP_limit = -19

In [ ]:
### CCMD Mock01

paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

for path in paths:
    data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
    boundary_mask = boundary_limit(data['ra'], data['dec'])
    data = data[boundary_mask]
    
    gal_pos = np.array([np.array(data['ra']),
                        np.array(data['dec']),    
                        np.array(data['z'])]).T
    gal_Mr  = np.array(data['Mr'])
    gal_gmr = np.array(data['gmr'])
    gal_z   = np.array(data['z'])

    env_LF_bottom20_red, env_LF_bottom20_blue, \
    env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = -19)

    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
### CCMD 4 Quads cov
paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

bot_red_data  = []
bot_blue_data = []
top_red_data  = []
top_blue_data = []

for path in paths:
    bottom_red_temp  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_bottom20_red_env_LF.npy')
    bot_red_data.append(bottom_red_temp)
    bottom_blue_temp = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_bottom20_blue_env_LF.npy')
    bot_blue_data.append(bottom_blue_temp)  

    top_red_temp     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_top20_red_env_LF.npy')
    top_red_data.append(top_red_temp)
    top_blue_temp    = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit19_Mr_C_Ng8_top20_blue_env_LF.npy')
    top_blue_data.append(top_blue_temp)

bot_red_data  = np.array(bot_red_data)
bot_blue_data = np.array(bot_blue_data)
top_red_data  = np.array(top_red_data)
top_blue_data = np.array(top_blue_data)

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' , np.mean(bot_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy', np.mean(bot_blue_data, axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_top20_red_env_LF_mean.npy'    , np.mean(top_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   , np.mean(top_blue_data, axis=0))

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' , np.cov(bot_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy', np.cov(bot_blue_data, rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_top20_red_env_LF_cov.npy'    , np.cov(top_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit19_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   , np.cov(top_blue_data, rowvar=False))

##### DDP_limit = -18

In [ ]:
### CCMD Mock01

paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

for path in paths:
    data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
    boundary_mask = boundary_limit(data['ra'], data['dec'])
    data = data[boundary_mask]
    
    gal_pos = np.array([np.array(data['ra']),
                        np.array(data['dec']),    
                        np.array(data['z'])]).T
    gal_Mr  = np.array(data['Mr'])
    gal_gmr = np.array(data['gmr'])
    gal_z   = np.array(data['z'])

    env_LF_bottom20_red, env_LF_bottom20_blue, \
    env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = -18)

    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
    np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
### CCMD 4 Quads cov
paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}

bot_red_data  = []
bot_blue_data = []
top_red_data  = []
top_blue_data = []

for path in paths:
    bottom_red_temp  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_bottom20_red_env_LF.npy')
    bot_red_data.append(bottom_red_temp)
    bottom_blue_temp = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_bottom20_blue_env_LF.npy')
    bot_blue_data.append(bottom_blue_temp)  

    top_red_temp     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_top20_red_env_LF.npy')
    top_red_data.append(top_red_temp)
    top_blue_temp    = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_' + path + '_DDP_limit18_Mr_C_Ng8_top20_blue_env_LF.npy')
    top_blue_data.append(top_blue_temp)

bot_red_data  = np.array(bot_red_data)
bot_blue_data = np.array(bot_blue_data)
top_red_data  = np.array(top_red_data)
top_blue_data = np.array(top_blue_data)

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' , np.mean(bot_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy', np.mean(bot_blue_data, axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_top20_red_env_LF_mean.npy'    , np.mean(top_red_data , axis=0))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   , np.mean(top_blue_data, axis=0))

np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' , np.cov(bot_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy', np.cov(bot_blue_data, rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_top20_red_env_LF_cov.npy'    , np.cov(top_red_data , rowvar=False))
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_DDP_limit18_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   , np.cov(top_blue_data, rowvar=False))

#### zip CCMD 4Quads stat&cov

In [ ]:
import numpy as np

# 1. 配置文件路径前缀和 DDP 对应的 limit 标号
base_dir = './CCMD_Main_4Quads_env_LF_DDP/'

ddp_limits = {
    'DDP18': '18',
    'DDP19': '19',
    'DDP20': '20'
}

# 2. 映射新键名到原文件名中的关键词
subset_map = {
    'bot_red':  'bottom20_red',
    'bot_blue': 'bottom20_blue',
    'top_red':  'top20_red',
    'top_blue': 'top20_blue'
}

# 3. 初始化两个顶层数据字典
CCMD_4Quads_stat = {}
CCMD_4Quads_cov  = {}

# 4. 遍历提取数据并组装字典
for ddp, limit in ddp_limits.items():
    CCMD_4Quads_stat[ddp] = {}
    CCMD_4Quads_cov[ddp]  = {}
    
    for new_key, old_key in subset_map.items():
        # 拼接文件名
        # 例如: CCMD_4Quads_DDP_limit18_Mr_C_Ng8_bottom20_red_env_LF_mean.npy
        mean_file = f"{base_dir}CCMD_4Quads_DDP_limit{limit}_Mr_C_Ng8_{old_key}_env_LF_mean.npy"
        cov_file  = f"{base_dir}CCMD_4Quads_DDP_limit{limit}_Mr_C_Ng8_{old_key}_env_LF_cov.npy"
        
        # 加载并存入字典
        CCMD_4Quads_stat[ddp][new_key] = np.load(mean_file)
        CCMD_4Quads_cov[ddp][new_key]  = np.load(cov_file)

# 5. 存储为两个独立的文件
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', CCMD_4Quads_stat, allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy',  CCMD_4Quads_cov,  allow_pickle=True)

print("CCMD 4Quads 的 2 个字典文件已成功保存！")

### zip CCMD 32Quads stat&cov, zip SDSS stat&cov

In [ ]:
import numpy as np

# 1. 配置文件路径与对应的简写键名
ddp_configs = {
    'DDP18': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/',
    'DDP19': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/',
    'DDP20': './'
}

# 映射原文件名中的关键词到您需要的新键名
subset_map = {
    'bot_red':  'bottom20_red',
    'bot_blue': 'bottom20_blue',
    'top_red':  'top20_red',
    'top_blue': 'top20_blue'
}

# 2. 初始化四个顶层数据字典
CCMD_stat, CCMD_cov, SDSS_stat, SDSS_cov = {}, {}, {}, {}

# 3. 遍历填充字典
for ddp, path in ddp_configs.items():
    CCMD_stat[ddp] = {}
    CCMD_cov[ddp]  = {}
    SDSS_stat[ddp] = {}
    SDSS_cov[ddp]  = {}
    
    # 提前加载 SDSS 的 npz cov 文件
    sdss_cov_raw = np.load(f"{path}SDSS_bright1_Mr_C_Ng8_env_LF.npz")
    
    for new_key, old_key in subset_map.items():
        ccmd_pref = 'CCMD_32Quads_Mr_C_Ng8_'
        sdss_pref = 'SDSS_bright1_Mr_C_Ng8_'
        
        # 存入 CCMD 数据
        CCMD_stat[ddp][new_key] = np.load(f"{path}{ccmd_pref}{old_key}_env_LF_mean.npy")
        CCMD_cov[ddp][new_key]  = np.load(f"{path}{ccmd_pref}{old_key}_env_LF_cov.npy")
        
        # 存入 SDSS 均值数据
        SDSS_stat[ddp][new_key] = np.load(f"{path}{sdss_pref}{old_key}_env_LF.npy")
        
        # 提取 SDSS 的 cov 数据（原 npz 中的键名为 cov_bottom_red 等形式）
        cov_key = f"cov_{old_key.replace('20', '')}"
        SDSS_cov[ddp][new_key]  = sdss_cov_raw[cov_key]

# 4. 存储为四个独立的文件 (使用 item() 保存字典结构)
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', CCMD_stat, allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy',  CCMD_cov,  allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/SDSS_stat_dict.npy', SDSS_stat, allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/SDSS_cov_dict.npy',  SDSS_cov,  allow_pickle=True)

print("4个字典文件已成功保存！")

#### zip CCMD 20Vars rsd stat&cov

In [ ]:
import numpy as np

# 1. 配置 20Vars 在不同 DDP 下的文件目录路径
ddp_paths = {
    'DDP18': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/',
    'DDP19': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/',
    'DDP20': './CCMD_20Vars_env_LF/'
}

# 2. 映射新字典键名到原文件名中的关键词
subset_map = {
    'bot_red':  'bottom20_red',
    'bot_blue': 'bottom20_blue',
    'top_red':  'top20_red',
    'top_blue': 'top20_blue'
}

# 3. 初始化 20Vars 的数据字典
CCMD_20Vars_stat = {}
CCMD_20Vars_cov  = {}

# 4. 遍历读取文件、计算统计量并存入字典
for ddp, path in ddp_paths.items():
    CCMD_20Vars_stat[ddp] = {}
    CCMD_20Vars_cov[ddp]  = {}
    
    for new_key, old_key in subset_map.items():
        # 拼接出完整的文件路径
        file_path = f"{path}CCMD_vars_box_rsd_env_LF_{old_key}.npy"
        
        # 加载包含 20 个 Vars 结果的原始二维数组
        # 形状如 (20, N_bins)，第一维是 realizations，第二维是 bins
        raw_data = np.load(file_path)
        
        # 计算均值 (stat)：沿着 axis=0 (即对 20 个 Var) 求平均
        CCMD_20Vars_stat[ddp][new_key] = np.mean(raw_data, axis=0)
        
        # 计算协方差矩阵 (cov)：rowvar=False 意味着列是不同的 bins 变量，行是观测(Var)
        CCMD_20Vars_cov[ddp][new_key] = np.cov(raw_data, rowvar=False)

# 5. 保存为标准的 .npy 字典文件
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', CCMD_20Vars_stat, allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy',  CCMD_20Vars_cov,  allow_pickle=True)

print("CCMD 20Vars 的 stat 和 cov 字典文件已成功保存！")

#### zip CCMD box w & w/o rsd stat

In [ ]:
import numpy as np

# 1. 配置文件路径（与之前 DDP 对应的目录保持一致，DDP20 在当前目录 './'）
ddp_paths = {
    'DDP18': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/',
    'DDP19': './CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/',
    'DDP20': './'
}

# 2. 映射新键名到原文件名中的关键词
subset_map = {
    'bot_red':  'bottom20_red',
    'bot_blue': 'bottom20_blue',
    'top_red':  'top20_red',
    'top_blue': 'top20_blue'
}

# 3. 初始化两个数据字典
CCMD_box_stat = {}
CCMD_box_rsd_stat = {}

# 4. 遍历读取文件并存入字典
for ddp, path in ddp_paths.items():
    CCMD_box_stat[ddp] = {}
    CCMD_box_rsd_stat[ddp] = {}
    
    for new_key, old_key in subset_map.items():
        # 拼接 box 和 box_rsd 的文件路径
        box_file = f"{path}CCMD_box_env_LF_{old_key}.npy"
        box_rsd_file = f"{path}CCMD_box_rsd_env_LF_{old_key}.npy"
        
        # 加载数据并存入字典
        CCMD_box_stat[ddp][new_key] = np.load(box_file)
        CCMD_box_rsd_stat[ddp][new_key] = np.load(box_rsd_file)

# 5. 保存为标准的 .npy 字典文件
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_stat_dict.npy', CCMD_box_stat, allow_pickle=True)
np.save('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_rsd_stat_dict.npy', CCMD_box_rsd_stat, allow_pickle=True)

print("CCMD box 和 box_rsd 的字典文件已成功保存！")

### Plot 4Quads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ==========================================
# 1. 加载数据
# ==========================================
# 使用 .item() 将 numpy object array 还原为字典
CCMD_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy',  allow_pickle=True).item()
SDSS_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_stat_dict.npy', allow_pickle=True).item()
SDSS_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_cov_dict.npy',  allow_pickle=True).item()

# CCMD_4Quads_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', allow_pickle=True).item()
# CCMD_4Quads_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy', allow_pickle=True).item()
CCMD_20Vars_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy', allow_pickle=True).item()

# ==========================================
# 2. 画布初始化
# ==========================================
fig = plt.figure(figsize=(15, 12), dpi=100)
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

ax_main  = [[None]*3 for _ in range(2)]
ax_ratio = [[None]*3 for _ in range(2)]

for i in range(2): 
    for j in range(3):
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], height_ratios=[3, 1], hspace=0)
        ax_main[i][j]  = fig.add_subplot(gs_cell[0])
        ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

# ==========================================
# 3. 核心绘图逻辑
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]             # (颜色标识, 行索引)
level_colors = [('bot', 'forestgreen'), ('top', 'orange')] # (前缀标识, 绘图颜色)
percent = 100

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, c_hex in level_colors:
            sub = f"{lvl}_{c_name}" # 拼接得到 bot_red, top_blue 等键名
            
            # --- 读取对应的4个核心结构 ---
            # stat      = CCMD_stat[ddp][sub]
            # cov       = CCMD_cov[ddp][sub]
            stat = CCMD_4Quads_stat[ddp][sub]
            cov  = CCMD_4Quads_cov[ddp][sub]

            stat_sdss = SDSS_stat[ddp][sub]
            cov_sdss  = SDSS_cov[ddp][sub]
            
            # --- 误差计算 ---
            yerr       = np.sqrt(cov.diagonal())
            yerr_sdss  = np.sqrt(cov_sdss.diagonal())
            yerr_total = np.sqrt((cov + CCMD_20Vars_cov[ddp][sub]).diagonal())

            # --- 主图 (ax_main) ---
            ax_main[i][j].plot(Mr_bins_x, stat, color=c_hex, alpha=0.9, lw=3, ls='--')
            ax_main[i][j].fill_between(Mr_bins_x, stat - yerr, stat + yerr, color=c_hex, alpha=0.3)
            ax_main[i][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color='royalblue', alpha=0.2)
            ax_main[i][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color='royalblue', alpha=0.2)
            
            ax_main[i][j].errorbar(Mr_bins_x, stat_sdss, yerr=yerr_sdss, color=c_hex, lw=2.5, alpha=1, ls='none', 
                                   capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

            # --- 比值图 (ax_ratio) ---
            ax_ratio[i][j].plot(Mr_bins_x, (stat/stat - 1)*percent, color=c_hex, alpha=0.9, lw=2, ls='-')
            ax_ratio[i][j].fill_between(Mr_bins_x, ((stat - yerr)/stat - 1)*percent, 
                                        ((stat + yerr)/stat - 1)*percent, color=c_hex, alpha=0.3)
            ax_ratio[i][j].fill_between(Mr_bins_x, ((stat - yerr_total)/stat - 1)*percent, 
                                        ((stat - yerr)/stat - 1)*percent, color='royalblue', alpha=0.2)
            ax_ratio[i][j].fill_between(Mr_bins_x, ((stat + yerr)/stat - 1)*percent, 
                                        ((stat + yerr_total)/stat - 1)*percent, color='royalblue', alpha=0.2)
            
            ax_ratio[i][j].errorbar(Mr_bins_x, (stat_sdss/stat - 1)*percent, yerr=(yerr_sdss/stat)*percent,
                                    color=c_hex, lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

# ==========================================
# 4. 图例与全局样式 
# ==========================================
# 提取第一组数据用于生成图例
dummy_stat = CCMD_stat['DDP18']['bot_red']
dummy_yerr = np.sqrt(CCMD_cov['DDP18']['bot_red'].diagonal())

# ax_main[0][0].plot(Mr_bins_x, dummy_stat - 1, color='black', alpha=0.9, lw=3, ls='--', label='CCMD')
ax_main[0][0].plot(Mr_bins_x, dummy_stat - 1, color='black', alpha=0.9, lw=3, ls='--', label='CCMD 4Quads')

ax_main[0][0].errorbar(Mr_bins_x, dummy_stat - 1, yerr=dummy_yerr, color='black', lw=2.5, alpha=1, ls='none', 
                       capsize=0, marker='o', ms=6, mec='black', mew=2.5, mfc='white', label='SDSS')
ax_main[1][0].plot(Mr_bins_x, dummy_stat - 1, color='forestgreen', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ bot20%')
ax_main[1][0].plot(Mr_bins_x, dummy_stat - 1, color='orange', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')

legend_0 = ax_main[0][0].legend(fontsize=22, loc='lower left', frameon=False) 
legend_1 = ax_main[1][0].legend(fontsize=20, loc='lower left', frameon=False) 
legend_1.get_texts()[0].set_color('forestgreen')
legend_1.get_texts()[1].set_color('orange')

# 全局样式处理
for i in range(2):
    for j in range(3):
        for loc, spine in ax_main[i][j].spines.items():  spine.set_linewidth(1 if loc == 'bottom' else 2)
        for loc, spine in ax_ratio[i][j].spines.items(): spine.set_linewidth(1 if loc == 'top' else 2)

        ax_main[i][j].set(xlim=(-22.5, -17.5), ylim=(10**-5.4, 10**-2.2), yscale='log')
        ax_main[i][j].tick_params(axis='both', direction='in', which='major', width=2, length=6, labelsize=24)
        ax_main[i][j].tick_params(axis='x', direction='in', which='minor', width=2, length=4, labelsize=24)
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)
        ax_main[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.4))
        
        ax_ratio[i][j].set(xlim=(-22.5, -17.5), ylim=(-40, 40), yticks=[-20, 20], yticklabels=['-20%', '20%'])
        ax_ratio[i][j].tick_params(axis='both', direction='in', which='both', width=2, length=6, labelsize=24)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4, labelsize=24)
        ax_ratio[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.4))
        ax_ratio[i][j].axhline(y=0, color='gray', linestyle='--', alpha=0.6)

        if j > 0: ax_main[i][j].set_yticklabels([]); ax_ratio[i][j].set_yticklabels([])
        if i == 0: ax_ratio[i][j].set_xticklabels([])

        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

ax_ratio[0][0].text(-21.5, 10, 'Diff.', fontsize=20)
for j in range(3):
    ax_main[0][j].text(-21.4, 1.0e-3, 'Red', color='red', fontsize=24)
    ax_main[1][j].text(-21.3, 2.0e-3, 'Blue', color='royalblue', fontsize=24)

titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']
for j in range(3):
    ax_main[0][j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)

fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=32, fontweight='bold', x=0.02)
fig.supxlabel(r'$M_r$', fontsize=32, fontweight='bold', y=0.03)

plt.tight_layout()
plt.show()

###  Plot 32 Quads

In [ ]:
# 32 Quads plot

### Plot 4Qauds in Mr [-22, -21]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker

# ==========================================
# 1. 极简加载：统一的字典数据结构
# ==========================================
SDSS_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_stat_dict.npy', allow_pickle=True).item()
SDSS_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_cov_dict.npy',  allow_pickle=True).item()

CCMD_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy',  allow_pickle=True).item()

CCMD_4Quads_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', allow_pickle=True).item()
CCMD_4Quads_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy',  allow_pickle=True).item()

CCMD_20Vars_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy',  allow_pickle=True).item()

# ==========================================
# 2. 画布与子图初始化
# ==========================================
fig = plt.figure(figsize=(15, 12), dpi=100)
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

ax_main  = [[None]*3 for _ in range(2)]
ax_ratio = [[None]*3 for _ in range(2)]

for i in range(2): 
    for j in range(3):
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], height_ratios=[3, 1], hspace=0)
        ax_main[i][j]  = fig.add_subplot(gs_cell[0])
        ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

# ==========================================
# 3. 核心绘图逻辑 (含数据拼接)
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]             
level_colors = [('bot', 'forestgreen'), ('top', 'orange')] 
percent = 100
split_idx = 5  # 拼接截断点：前5个使用 4Quads

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, c_hex in level_colors:
            sub = f"{lvl}_{c_name}" 
            
            # --- 读取 SDSS 数据 ---
            s_mean    = SDSS_stat[ddp][sub]
            yerr_sdss = np.sqrt(SDSS_cov[ddp][sub].diagonal())
            
            # --- 读取对应的 CCMD 和 20Vars 结构 ---
            c4_mean  = CCMD_4Quads_stat[ddp][sub]
            c4_cov   = CCMD_4Quads_cov[ddp][sub]
            
            c32_mean = CCMD_stat[ddp][sub]
            c32_cov  = CCMD_cov[ddp][sub]
            
            cov_20v  = CCMD_20Vars_cov[ddp][sub]
            
            # ================= 重点：拼接逻辑 =================
            # 1. 拼接均值 (stat)
            c_mean = np.concatenate([c4_mean[:split_idx], c32_mean[split_idx:]])
            
            # 2. 拼接基础误差 (从对角线提取 std)
            std_c4  = np.sqrt(c4_cov.diagonal())[:split_idx]
            std_c32 = np.sqrt(c32_cov.diagonal())[split_idx:]
            yerr    = np.concatenate([std_c4, std_c32])
            
            # 3. 拼接总误差 (协方差相加提取对角线，或对角线相加开根)
            std_tot_c4  = np.sqrt((c4_cov + cov_20v).diagonal())[:split_idx]
            std_tot_c32 = np.sqrt((c32_cov + cov_20v).diagonal())[split_idx:]
            yerr_total  = np.concatenate([std_tot_c4, std_tot_c32])
            # ==================================================

            # --- 主图 (ax_main) ---
            ax_main[i][j].plot(Mr_bins_x, c_mean, color=c_hex, alpha=0.9, lw=3, ls='--')
            ax_main[i][j].fill_between(Mr_bins_x, c_mean - yerr, c_mean + yerr, color=c_hex, alpha=0.3)
            ax_main[i][j].fill_between(Mr_bins_x, c_mean - yerr_total, c_mean - yerr, color='royalblue', alpha=0.2)
            ax_main[i][j].fill_between(Mr_bins_x, c_mean + yerr, c_mean + yerr_total, color='royalblue', alpha=0.2)
            
            ax_main[i][j].errorbar(Mr_bins_x, s_mean, yerr=yerr_sdss, color=c_hex, lw=2.5, alpha=1, ls='none', 
                                   capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

            # --- 比值图 (ax_ratio) ---
            ax_ratio[i][j].plot(Mr_bins_x, (c_mean/c_mean - 1)*percent, color=c_hex, alpha=0.9, lw=2, ls='-')
            ax_ratio[i][j].fill_between(Mr_bins_x, ((c_mean - yerr)/c_mean - 1)*percent, 
                                        ((c_mean + yerr)/c_mean - 1)*percent, color=c_hex, alpha=0.3)
            ax_ratio[i][j].fill_between(Mr_bins_x, ((c_mean - yerr_total)/c_mean - 1)*percent, 
                                        ((c_mean - yerr)/c_mean - 1)*percent, color='royalblue', alpha=0.2)
            ax_ratio[i][j].fill_between(Mr_bins_x, ((c_mean + yerr)/c_mean - 1)*percent, 
                                        ((c_mean + yerr_total)/c_mean - 1)*percent, color='royalblue', alpha=0.2)
            
            ax_ratio[i][j].errorbar(Mr_bins_x, (s_mean/c_mean - 1)*percent, yerr=(yerr_sdss/c_mean)*percent,
                                    color=c_hex, lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

# ==========================================
# 4. 图例与全局样式 (含整数竖线与相对位置文本)
# ==========================================
# 提取一个样本用于生成图例底板
dummy_mean = CCMD_4Quads_stat['DDP18']['bot_red'][0]
dummy_yerr = np.sqrt(CCMD_4Quads_cov['DDP18']['bot_red'].diagonal())[0]

ax_main[0][0].plot(Mr_bins_x[0], dummy_mean - 1, color='black', alpha=0.9, lw=3, ls='--', label='CCMD')
ax_main[0][0].errorbar(Mr_bins_x[0], dummy_mean - 1, yerr=dummy_yerr, color='black', lw=2.5, alpha=1, ls='none', 
                       capsize=0, marker='o', ms=6, mec='black', mew=2.5, mfc='white', label='SDSS')
ax_main[1][0].plot(Mr_bins_x[0], dummy_mean - 1, color='orange', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')
ax_main[1][0].plot(Mr_bins_x[0], dummy_mean - 1, color='forestgreen', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ bot20%')

legend_0 = ax_main[0][0].legend(fontsize=22, loc='lower left', frameon=False) 
legend_1 = ax_main[1][0].legend(fontsize=20, loc='lower left', frameon=False) 
legend_1.get_texts()[0].set_color('orange')
legend_1.get_texts()[1].set_color('forestgreen')

# ================= 新增：在 ax_main[0][0] 添加相对位置文本 =================
# transform=ax_main[0][0].transAxes 意味着 (0,0) 是左下角，(1,1) 是右上角
# 我将 underdense 稍微往下放了一点 (0.4)，避免两个词完全重叠，您可以自行修改
ax_main[1][0].text(0.52, 0.22, 'overdense', transform=ax_main[1][0].transAxes, 
                   fontsize=14, color='orange', ha='center', va='center')
ax_main[1][0].text(0.52, 0.06, 'underdense', transform=ax_main[1][0].transAxes, 
                   fontsize=14, color='forestgreen', ha='center', va='center')
# =========================================================================

# 全局样式循环设置
for i in range(2):
    for j in range(3):
        # 边框粗细
        for loc, spine in ax_main[i][j].spines.items():  spine.set_linewidth(1 if loc == 'bottom' else 2)
        for loc, spine in ax_ratio[i][j].spines.items(): spine.set_linewidth(1 if loc == 'top' else 2)

        # 主图刻度 (向内)
        ax_main[i][j].set(xlim=(-22.5, -17.5), ylim=(10**-5.4, 10**-2.2), yscale='log', )
        ax_main[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_main[i][j].tick_params(axis='both', which='major', width=3, length=8, labelsize=24)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)
        ax_main[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
        ax_main[i][j].xaxis.set_major_locator(ticker.MultipleLocator(1.0))
        
        # 比值图刻度 (向内)
        ax_ratio[i][j].set(xlim=(-22.5, -17.5), ylim=(-40, 40), yticks=[-20, 20], yticklabels=['-20%', '20%'],
                          xticks=[-18, -19, -20, -21, -22], xticklabels=['-18', '-19', '-20', '-21', '-22'])
        ax_ratio[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_ratio[i][j].tick_params(axis='both', which='major', width=3, length=8, labelsize=24)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_ratio[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
        ax_ratio[i][j].xaxis.set_major_locator(ticker.MultipleLocator(1.0))
        ax_ratio[i][j].axhline(y=0, color='gray', linestyle='--', alpha=0.6)

        # ================= 新增：在整数位置添加灰色竖线 =================
        for x_val in [-18, -19, -20, -21, -22]:
            ax_main[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)
            ax_ratio[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)
        # ================================================================

        # 隐藏相邻子图的重复标签
        if j > 0: 
            ax_main[i][j].set_yticklabels([])
            ax_ratio[i][j].set_yticklabels([])
        if i == 0: 
            ax_ratio[i][j].set_xticklabels([])

        # 反转 X 轴 (放在设置了刻度范围之后)
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

# 添加文本和标题
ax_ratio[0][0].text(-21.5, 10, 'Diff.', fontsize=20)
for j in range(3):
    ax_main[0][j].text(-21.4, 2.0e-3, 'Red', color='red', fontsize=24)
    ax_main[1][j].text(-21.3, 2.0e-3, 'Blue', color='royalblue', fontsize=24)

titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']
for j in range(3):
    # ax_main[0][j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)
    ax_main[0][j].set_title( titles[j], color='k', fontsize=22)

# 添加全局坐标轴标签
fig.supylabel(r'$\Phi_{\tilde{N}_g^8}~[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=32, fontweight='bold', x=0.02)
fig.supxlabel(r'$M_r$', fontsize=32, fontweight='bold', y=0.03)

plt.tight_layout()
plt.show()

### Plot 32Quads with box

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ==========================================
# 1. 极简加载：直接读取格式化好的字典
# ==========================================
CCMD_stat         = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov          = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy', allow_pickle=True).item()
CCMD_box_stat     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_stat_dict.npy', allow_pickle=True).item()
CCMD_box_rsd_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_rsd_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy', allow_pickle=True).item()

# ==========================================
# 2. 画布初始化
# ==========================================
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True, dpi=100)
plt.subplots_adjust(hspace=0, wspace=0)

# ==========================================
# 3. 核心绘图循环 (自动遍历行列和 top/bot)
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]
levels = [('bot', '--'), ('top', '-')]  # 绑定前后缀和对应的线型

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, ls in levels:
            sub = f"{lvl}_{c_name}"  # 拼装出 bot_red, top_blue 等键名
            
            # --- 1. CCMD 32 quads ---
            stat_32 = CCMD_stat[ddp][sub]
            yerr_32 = np.sqrt(CCMD_cov[ddp][sub].diagonal())
            ax[i, j].plot(Mr_bins_x, stat_32, color='forestgreen', alpha=0.9, lw=3, ls=ls)
            ax[i, j].fill_between(Mr_bins_x, stat_32 - yerr_32, stat_32 + yerr_32, color='forestgreen', alpha=0.3)
            
            # --- 2. CCMD box ---
            stat_box = CCMD_box_stat[ddp][sub]
            ax[i, j].plot(Mr_bins_x, stat_box, color='black', alpha=0.9, lw=3, ls=ls)
            
            # --- 3. CCMD box rsd ---
            stat_box_rsd = CCMD_box_rsd_stat[ddp][sub]
            ax[i, j].plot(Mr_bins_x, stat_box_rsd, color='red', alpha=0.9, lw=3, ls=ls)
            
            # --- 4. CCMD 20 Vars ---
            stat_20v = CCMD_20Vars_stat[ddp][sub]
            yerr_20v = np.sqrt(CCMD_20Vars_cov[ddp][sub].diagonal())
            ax[i, j].plot(Mr_bins_x, stat_20v, color='royalblue', alpha=0.9, lw=3, ls=ls)
            ax[i, j].fill_between(Mr_bins_x, stat_20v - yerr_20v, stat_20v + yerr_20v, color='royalblue', alpha=0.2)

# ==========================================
# 4. 图例底板与添加 (巧妙隐藏绘图点)
# ==========================================
# 借用一个不在显示范围内的 Y 值来画图例线
dummy_y = CCMD_stat['DDP18']['bot_red'] - 1 

# Top/Bottom 图例 (画在 ax[0, 0])
ax[0, 0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')
ax[0, 0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='--', label=r'$N_g^8$ bot20%')

# 数据类别图例 (画在 ax[1, 0])
ax[1, 0].plot(Mr_bins_x, dummy_y, color='forestgreen', alpha=0.9, lw=3, ls='-', label='CCMD')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='black', alpha=0.9, lw=3, ls='-', label='CCMD box w/o rsd')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='red', alpha=0.9, lw=3, ls='-', label='CCMD box w/ rsd')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='royalblue', alpha=0.9, lw=3, ls='-', label='CCMD 20Vars box w/ rsd')

legend_0 = ax[0, 0].legend(fontsize=22, loc='lower left', frameon=False)
legend_1 = ax[1, 0].legend(fontsize=14, loc='lower left', frameon=False)
legend_1.get_texts()[0].set_color('forestgreen')
legend_1.get_texts()[2].set_color('red')
legend_1.get_texts()[3].set_color('royalblue')

# ==========================================
# 5. 全局样式与刻度设置
# ==========================================
for i in range(2):
    for j in range(3):
        # 边框粗细统一设为2
        for spine in ax[i, j].spines.values():
            spine.set_linewidth(2)

        # 刻度与范围
        ax[i, j].set_xlim(-22.5, -17.5)
        ax[i, j].set_ylim(10**-5.4, 10**-2.2)
        ax[i, j].set_yscale('log')
        
        ax[i, j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax[i, j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax[i, j].tick_params(axis='x', which='minor', width=2, length=4)
        ax[i, j].tick_params(axis='y', which='minor', length=0)
        ax[i, j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))

        # ================= 新增：在整数位置添加灰色竖线 =================
        for x_val in [-18, -19, -20, -21, -22]:
            ax[i, j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)
        # ================================================================

# 由于使用了 sharex=True，翻转任何一个子图的 X 轴即可翻转所有
ax[0, 0].invert_xaxis()

# ==========================================
# 6. 添加文本标签
# ==========================================
titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']
for j in range(3):
    ax[0, j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)
    ax[0, j].text(-21.4, 1e-3, 'Red', color='red', fontsize=24)
    ax[1, j].text(-21.3, 1.5e-3, 'Blue', color='royalblue', fontsize=24)
    # ================= 新增：在 ax_main[0][0] 添加相对位置文本 =================
    # transform=ax_main[0][0].transAxes 意味着 (0,0) 是左下角，(1,1) 是右上角
    # 我将 underdense 稍微往下放了一点 (0.4)，避免两个词完全重叠，您可以自行修改
    ax[0, 0].text(0.3, 0.82, 'overdense', transform=ax[0, 0].transAxes, 
                    fontsize=16, color='gray', ha='center', va='center')
    ax[0, 0].text(0.3, 0.58, 'underdense', transform=ax[0, 0].transAxes, 
                    fontsize=16, color='gray', ha='center', va='center')
# =========================================================================
fig.supylabel(r'$\tilde{\Phi}_{N_g^8}~[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=30, fontweight='bold', x=0.02)
fig.supxlabel(r'$M_r$', fontsize=30, fontweight='bold', y=0.02)

plt.show()

### Plot 4Quads in Mr [-22,-21] with box

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ==========================================
# 1. 极简加载：直接读取格式化好的字典
# ==========================================
CCMD_stat         = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov          = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy', allow_pickle=True).item()

# 增加加载 4Quads 数据用于拼接
CCMD_4Quads_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', allow_pickle=True).item()
CCMD_4Quads_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy', allow_pickle=True).item()

CCMD_box_stat     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_stat_dict.npy', allow_pickle=True).item()
CCMD_box_rsd_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_rsd_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy', allow_pickle=True).item()

# ==========================================
# 2. 画布初始化
# ==========================================
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True, dpi=100)
plt.subplots_adjust(hspace=0, wspace=0)

# ==========================================
# 3. 核心绘图循环 (自动遍历行列和 top/bot)
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]
levels = [('bot', '--'), ('top', '-')]  # 绑定前后缀和对应的线型
split_idx = 5  # 拼接截断点：前5个使用 4Quads

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, ls in levels:
            sub = f"{lvl}_{c_name}"  # 拼装出 bot_red, top_blue 等键名
            
            # --- 1. CCMD (Stitched 4Quads + 32Quads) ---
            # 提取 32Quads 数据
            stat_32 = CCMD_stat[ddp][sub]
            yerr_32 = np.sqrt(CCMD_cov[ddp][sub].diagonal())
            
            # 提取 4Quads 数据
            stat_4  = CCMD_4Quads_stat[ddp][sub]
            yerr_4  = np.sqrt(CCMD_4Quads_cov[ddp][sub].diagonal())
            
            # 拼接均值和误差
            stat_stitched = np.concatenate([stat_4[:split_idx], stat_32[split_idx:]])
            yerr_stitched = np.concatenate([yerr_4[:split_idx], yerr_32[split_idx:]])
            
            ax[i, j].plot(Mr_bins_x, stat_stitched, color='forestgreen', alpha=0.9, lw=3, ls=ls)
            ax[i, j].fill_between(Mr_bins_x, stat_stitched - yerr_stitched, stat_stitched + yerr_stitched, 
                                  color='forestgreen', alpha=0.3)
            
            # --- 2. CCMD box (不动) ---
            stat_box = CCMD_box_stat[ddp][sub]
            ax[i, j].plot(Mr_bins_x, stat_box, color='black', alpha=0.9, lw=3, ls=ls)
            
            # --- 3. CCMD box rsd (不动) ---
            stat_box_rsd = CCMD_box_rsd_stat[ddp][sub]
            ax[i, j].plot(Mr_bins_x, stat_box_rsd, color='red', alpha=0.9, lw=3, ls=ls)
            
            # --- 4. CCMD 20 Vars (不动) ---
            stat_20v = CCMD_20Vars_stat[ddp][sub]
            yerr_20v = np.sqrt(CCMD_20Vars_cov[ddp][sub].diagonal())
            ax[i, j].plot(Mr_bins_x, stat_20v, color='royalblue', alpha=0.9, lw=3, ls=ls)
            ax[i, j].fill_between(Mr_bins_x, stat_20v - yerr_20v, stat_20v + yerr_20v, color='royalblue', alpha=0.2)

# ==========================================
# 4. 图例底板与添加 (巧妙隐藏绘图点)
# ==========================================
# 借用一个不在显示范围内的 Y 值来画图例线
dummy_y = CCMD_stat['DDP18']['bot_red'] - 1 

# Top/Bottom 图例 (画在 ax[0, 0])
ax[0, 0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')
ax[0, 0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='--', label=r'$N_g^8$ bot20%')

# 数据类别图例 (画在 ax[1, 0])
ax[1, 0].plot(Mr_bins_x, dummy_y, color='forestgreen', alpha=0.9, lw=3, ls='-', label='CCMD')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='black', alpha=0.9, lw=3, ls='-', label='CCMD box w/o rsd')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='red', alpha=0.9, lw=3, ls='-', label='CCMD box w/ rsd')
ax[1, 0].plot(Mr_bins_x, dummy_y, color='royalblue', alpha=0.9, lw=3, ls='-', label='CCMD 20Vars box w/ rsd')

legend_0 = ax[0, 0].legend(fontsize=22, loc='lower left', frameon=False)
legend_1 = ax[1, 0].legend(fontsize=14, loc='lower left', frameon=False)
legend_1.get_texts()[0].set_color('forestgreen')
legend_1.get_texts()[2].set_color('red')
legend_1.get_texts()[3].set_color('royalblue')

# ==========================================
# 5. 全局样式与刻度设置
# ==========================================
for i in range(2):
    for j in range(3):
        # 边框粗细统一设为2
        for spine in ax[i, j].spines.values():
            spine.set_linewidth(2)

        # 刻度与范围
        ax[i, j].set(xlim=(-22.5, -17.5), ylim=(10**-5.4, 10**-2.2),
                        xticks=[-18, -19, -20, -21, -22], xticklabels=['-18', '-19', '-20', '-21', '-22'])
        ax[i, j].set_yscale('log')
        ax[i, j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax[i, j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax[i, j].tick_params(axis='x', which='minor', width=2, length=4)
        ax[i, j].tick_params(axis='y', which='minor', length=0)
        ax[i, j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))
        ax[i][j].xaxis.set_major_locator(ticker.MultipleLocator(1.0))

        # 在整数位置添加灰色竖线
        for x_val in [-18, -19, -20, -21, -22]:
            ax[i, j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)

# 由于使用了 sharex=True，翻转任何一个子图的 X 轴即可翻转所有
ax[0, 0].invert_xaxis()

# ==========================================
# 6. 添加文本标签
# ==========================================
titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']

for j in range(3):
    # ax[0, j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)
    ax[0, j].set_title(titles[j], color='k', fontsize=22)
    ax[0, j].text(-21.4, 2e-3, 'Red', color='red', fontsize=24)
    ax[1, j].text(-21.3, 2e-3, 'Blue', color='royalblue', fontsize=24)

# 相对位置文本（移出循环，避免重复书写）
ax[0, 0].text(0.55, 0.21, 'overdense', transform=ax[0, 0].transAxes, 
                   fontsize=14, color='gray', ha='center', va='center')
ax[0, 0].text(0.55, 0.06, 'underdense', transform=ax[0, 0].transAxes, 
                   fontsize=14, color='gray', ha='center', va='center')

fig.supylabel(r'$\Phi_{\tilde{N}_g^8}~[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=30, fontweight='bold', x=0.03)
fig.supxlabel(r'$M_r$', fontsize=30, fontweight='bold', y=0.02)

plt.show()

#### Plot 4Quads in Mr [-22,-21], ratio for top

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker

# 假设 Mr_bins_x 已经在外部定义，长度应为20
# Mr_bins_x = np.linspace(-22.5, -17.5, 20) 

# ==========================================
# 1. 加载所有字典数据 (含新加载的 CCMD_4Quads)
# ==========================================
# SDSS_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_stat_dict.npy', allow_pickle=True).item()
# SDSS_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_cov_dict.npy', allow_pickle=True).item()

CCMD_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy', allow_pickle=True).item()

# 新增加载 4Quads 数据用于拼接
CCMD_4Quads_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', allow_pickle=True).item()
CCMD_4Quads_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy', allow_pickle=True).item()

CCMD_box_stat     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_stat_dict.npy', allow_pickle=True).item()
CCMD_box_rsd_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_rsd_stat_dict.npy', allow_pickle=True).item()

CCMD_20Vars_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy', allow_pickle=True).item()

# ==========================================
# 2. 画布与子图初始化 (含 nested GridSpec)
# ==========================================
fig = plt.figure(figsize=(15, 12), dpi=100)
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

ax_main  = [[None]*3 for _ in range(2)]
ax_ratio = [[None]*3 for _ in range(2)]

for i in range(2): 
    for j in range(3):
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], height_ratios=[3, 1], hspace=0)
        ax_main[i][j]  = fig.add_subplot(gs_cell[0])
        ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

# ==========================================
# 3. 核心绘图循环
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]             
levels = [('bot', '--'), ('top', '-')]  # 绑定前后缀和对应的线型
percent = 100
split_idx = 5  # 拼接截断点：前5个使用 4Quads 数据

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, ls in levels:
            sub = f"{lvl}_{c_name}" 

            # --- 读取 SDSS 数据 ---
            # s_mean = SDSS_stat[ddp][sub]
            # s_cov  = SDSS_cov[ddp][sub]
            # yerr_sdss = np.sqrt(s_cov.diagonal())

            # --- 读取 CCMD 4Quads 和 32Quads 数据 ---
            c4_mean  = CCMD_4Quads_stat[ddp][sub]
            c4_cov   = CCMD_4Quads_cov[ddp][sub]
            c32_mean = CCMD_stat[ddp][sub]
            c32_cov  = CCMD_cov[ddp][sub]
            # 为了画图代码的简洁，临时创建一个对角误差映射
            std_stitched_map = {
                ddp: {sub: np.sqrt((CCMD_cov[ddp][sub]).diagonal()) for sub in CCMD_cov[ddp]}
            }

            # ================= 重点：拼接逻辑 =================
            # 1. 拼接均值 (Mean)
            c_mean = np.concatenate([c4_mean[:split_idx], c32_mean[split_idx:]])

            # 2. 拼接误差 (从对角线提取 std)
            std_c4  = np.sqrt(c4_cov.diagonal())[:split_idx]
            std_c32 = np.sqrt(c32_cov.diagonal())[split_idx:]
            yerr    = np.concatenate([std_c4, std_c32])
            # ==================================================

            # --- 主图 (ax_main) ---
            ax_main[i][j].plot(Mr_bins_x, c_mean, color=c_hex, alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].fill_between(Mr_bins_x, c_mean - yerr, c_mean + yerr, color=c_hex, alpha=0.3)

            # ax_main[i][j].errorbar(Mr_bins_x, s_mean, yerr=yerr_sdss, color=c_hex, lw=2.5, alpha=1, ls='none',
                                #    capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

            # --- 4. CCMD box 和 Vars (不動) ---
            # 临时获取 20Vars 的 cov 对角误差用于 fill_between
            yerr_20v = np.sqrt((CCMD_20Vars_cov[ddp][sub]).diagonal())

            ax_main[i][j].plot(Mr_bins_x, CCMD_box_stat[ddp][sub], color='black', alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].plot(Mr_bins_x, CCMD_box_rsd_stat[ddp][sub], color='red', alpha=0.9, lw=3, ls=ls)

            ax_main[i][j].plot(Mr_bins_x, CCMD_20Vars_stat[ddp][sub], color='royalblue', alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].fill_between(Mr_bins_x, CCMD_20Vars_stat[ddp][sub] - yerr_20v, 
                                       CCMD_20Vars_stat[ddp][sub] + yerr_20v, color='royalblue', alpha=0.2)
            # ================= 新增：比值图数据计算与绘制 =================
            # 基准：Stitched CCMD Mean (c_mean)
            # 比值图仅展示 'top' 部分：
            # 在主图顶部 ax_main[0][j] (Red) 和 ax_main[1][j] (Blue) 下方的比值图中
            # 基准Denominator始终是该 color_name 下的拼接后的 stitched_top_c_mean

            # 1. 临时读取拼接后的 top 数据作为 Denominator 基准
            c4_mean_top  = CCMD_4Quads_stat[ddp][f'top_{c_name}']
            c32_mean_top = CCMD_stat[ddp][f'top_{c_name}']
            stitched_top_c_mean = np.concatenate([c4_mean_top[:split_idx], c32_mean_top[split_idx:]])
            denominator = stitched_top_c_mean

            # 临时读取拼接后的 CCMD Top Cov 对角std 作为 fill_between 误差范围底
            c4_cov_top  = CCMD_4Quads_cov[ddp][f'top_{c_name}']
            c32_cov_top = CCMD_cov[ddp][f'top_{c_name}']
            std_c4_top  = np.sqrt(c4_cov_top.diagonal())[:split_idx]
            std_c32_top = np.sqrt(c32_cov_top.diagonal())[split_idx:]
            stitched_top_c_yerr = np.concatenate([std_c4_top, std_c32_top])

            # 2. 绘制 CCMD Cov 基准误差填充 (Forestgreen)
            # 以 1.0 为基准线，填充归一化后的 stitched_top_c_yerr
            ax_ratio[i][j].fill_between(Mr_bins_x, 1.0 - (stitched_top_c_yerr / denominator), 
                                        1.0 + (stitched_top_c_yerr / denominator), color='forestgreen', alpha=0.3)
            ax_ratio[i][j].axhline(y=1.0, color='gray', linestyle='--', alpha=0.6) # 基准线

            # 3. 绘制 Numerator Ratios (solid lines only)
            # 均值部分：
            # Box (black)
            # Box RSD (red)
            # 20Vars (royalblue) 均值
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_box_stat[ddp][f'top_{c_name}'] / denominator, color='black', alpha=0.9, lw=3, ls=ls)
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_box_rsd_stat[ddp][f'top_{c_name}'] / denominator, color='red', alpha=0.9, lw=3, ls=ls)
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_20Vars_stat[ddp][f'top_{c_name}'] / denominator, color='royalblue', alpha=0.9, lw=3, ls=ls)
            # ================================================================

# ==========================================
# 4. 图例与全局样式 (含刻度修复、灰色竖线、相对位置文本)
# ==========================================
# 提取一个样本数据用于生成图例底板 (借用 dummy_y 确保不在显示范围)
dummy_y = CCMD_stat['DDP18']['bot_red'] - 1 

ax_main[0][0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')
ax_main[0][0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='--', label=r'$N_g^8$ bot20%')

ax_main[1][0].plot(Mr_bins_x, dummy_y, color='forestgreen', alpha=0.9, lw=3, ls='-', label='CCMD')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='black', alpha=0.9, lw=3, ls='-', label='CCMD box w/o rsd')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='red', alpha=0.9, lw=3, ls='-', label='CCMD box w/ rsd')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='royalblue', alpha=0.9, lw=3, ls='-', label='CCMD 20Vars box w/ rsd')

legend_0 = ax_main[0][0].legend(fontsize=22, loc='lower left', frameon=False)
legend_1 = ax_main[1][0].legend(fontsize=14, loc='lower left', frameon=False)
legend_1.get_texts()[0].set_color('forestgreen')
legend_1.get_texts()[2].set_color('red')
legend_1.get_texts()[3].set_color('royalblue')

# ================= 新增：在 ax_main[0][0] 添加相对位置文本 =================
# transform=ax_main[0][0].transAxes 意味着 (0,0) 是左下角，(1,1) 是右上角
# 我将 underdense 稍微往下放了一点 (0.4)，避免两个词完全重叠，您可以自行修改
ax_main[0][0].text(0.3, 0.82, 'overdense', transform=ax_main[0][0].transAxes,fontsize=16, color='gray', ha='center', va='center')
ax_main[0][0].text(0.3, 0.56, 'underdense', transform=ax_main[0][0].transAxes, fontsize=16, color='gray', ha='center', va='center')
# =========================================================================

# 全局样式循环设置
for i in range(2):
    for j in range(3):
        # 边框粗细统一设为2
        for spine in ax_main[i][j].spines.values(): spine.set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values(): spine.set_linewidth(2)

        # 主图刻度 (向内)
        ax_main[i][j].set(xlim=(-22.5, -17.5), ylim=(10**-5.4, 10**-2.2), yscale='log')
        ax_main[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)
        ax_main[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))

        # ================= 修改新增：比值图刻度与范围设置 =================
        ax_ratio[i][j].set(xlim=(-22.5, -17.5), ylim=(0.4, 1.6), yticks=[0.8, 1.2], yticklabels=['-20%', '20%']) # 调整比值图范围和刻度
		# 添加比值图文本 'Diff.' (gray, transform relative position)
		# 可以在循环中添加，也可以单独添加，这里统一添加
        ax_ratio[i][j].text(0.95, 0.85, 'Diff.', transform=ax_ratio[i][j].transAxes, 
                             fontsize=16, color='gray', ha='right', va='center')

        ax_ratio[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_ratio[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))
        # ================================================================

        # ================= 新增：在整数位置添加灰色竖线 =================
        for x_val in [-18, -19, -20, -21, -22]:
            ax_main[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)
            ax_ratio[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5) # 比值图也加灰色竖线
        # ================================================================

        # 隐藏多余标签
        if j > 0: ax_main[i][j].set_yticklabels([]); ax_ratio[i][j].set_yticklabels([])
        if i == 0: ax_ratio[i][j].set_xticklabels([])

        # 反转X轴 (放在设置了刻度范围之后)
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

# 文本与标题
for j in range(3):
    ax_main[0][j].text(-21.4, 1.0e-3, 'Red', color='red', fontsize=24)
    ax_main[1][j].text(-21.3, 1.5e-3, 'Blue', color='royalblue', fontsize=24)

titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']
for j in range(3):
    ax_main[0][j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)


# ======================================================

fig.supylabel(r'$\tilde{\Phi}_{N_g^8}~[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=30, fontweight='bold', x=0.02)
fig.supxlabel(r'$M_r$', fontsize=30, fontweight='bold', y=0.02)

plt.tight_layout()
plt.show()

### Plot 4Quads in Mr[-22,-21] for bot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker

# 假设 Mr_bins_x 已经在外部定义，长度应为20
# Mr_bins_x = np.linspace(-22.5, -17.5, 20) 

# ==========================================
# 1. 加载所有字典数据 (含新加载的 CCMD_4Quads)
# ==========================================
# SDSS_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_stat_dict.npy', allow_pickle=True).item()
# SDSS_cov  = np.load('./CCMD_Main_4Quads_env_LF_DDP/SDSS_cov_dict.npy', allow_pickle=True).item()

CCMD_stat         = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_stat_dict.npy', allow_pickle=True).item()
CCMD_cov          = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_cov_dict.npy', allow_pickle=True).item()

# 新增加载 4Quads 数据用于拼接
CCMD_4Quads_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_stat_dict.npy', allow_pickle=True).item()
CCMD_4Quads_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_4Quads_cov_dict.npy', allow_pickle=True).item()

CCMD_box_stat     = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_stat_dict.npy', allow_pickle=True).item()
CCMD_box_rsd_stat = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_box_rsd_stat_dict.npy', allow_pickle=True).item()

CCMD_20Vars_stat  = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_stat_dict.npy', allow_pickle=True).item()
CCMD_20Vars_cov   = np.load('./CCMD_Main_4Quads_env_LF_DDP/CCMD_20Vars_cov_dict.npy', allow_pickle=True).item()

# ==========================================
# 2. 画布与子图初始化 (含 nested GridSpec)
# ==========================================
fig = plt.figure(figsize=(15, 12), dpi=100)
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

ax_main  = [[None]*3 for _ in range(2)]
ax_ratio = [[None]*3 for _ in range(2)]

for i in range(2): 
    for j in range(3):
        gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], height_ratios=[3, 1], hspace=0)
        ax_main[i][j]  = fig.add_subplot(gs_cell[0])
        ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

# ==========================================
# 3. 核心绘图循环
# ==========================================
ddp_keys = ['DDP18', 'DDP19', 'DDP20']
color_rows = [('red', 0), ('blue', 1)]             
levels = [('bot', '--'), ('top', '-')]  # 绑定前后缀和对应的线型
percent = 100
split_idx = 5  # 拼接截断点：前5个使用 4Quads 数据

for j, ddp in enumerate(ddp_keys):
    for c_name, i in color_rows:
        for lvl, ls in levels:
            sub = f"{lvl}_{c_name}" 

            # --- 读取 SDSS 数据 ---
            # s_mean = SDSS_stat[ddp][sub]
            # s_cov  = SDSS_cov[ddp][sub]
            # yerr_sdss = np.sqrt(s_cov.diagonal())

            # --- 读取 CCMD 4Quads 和 32Quads 数据 ---
            c4_mean  = CCMD_4Quads_stat[ddp][sub]
            c4_cov   = CCMD_4Quads_cov[ddp][sub]
            c32_mean = CCMD_stat[ddp][sub]
            c32_cov  = CCMD_cov[ddp][sub]
            # 为了画图代码的简洁，临时创建一个对角误差映射
            std_stitched_map = {
                ddp: {sub: np.sqrt((CCMD_cov[ddp][sub]).diagonal()) for sub in CCMD_cov[ddp]}
            }

            # ================= 重点：拼接逻辑 =================
            # 1. 拼接均值 (Mean)
            c_mean = np.concatenate([c4_mean[:split_idx], c32_mean[split_idx:]])

            # 2. 拼接误差 (从对角线提取 std)
            std_c4  = np.sqrt(c4_cov.diagonal())[:split_idx]
            std_c32 = np.sqrt(c32_cov.diagonal())[split_idx:]
            yerr    = np.concatenate([std_c4, std_c32])
            # ==================================================

            # --- 主图 (ax_main) ---
            ax_main[i][j].plot(Mr_bins_x, c_mean, color=c_hex, alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].fill_between(Mr_bins_x, c_mean - yerr, c_mean + yerr, color=c_hex, alpha=0.3)

            # ax_main[i][j].errorbar(Mr_bins_x, s_mean, yerr=yerr_sdss, color=c_hex, lw=2.5, alpha=1, ls='none',
                                #    capsize=0, marker='o', ms=6, mec=c_hex, mew=2.5, mfc='white')

            # --- 4. CCMD box 和 Vars (不動) ---
            # 临时获取 20Vars 的 cov 对角误差用于 fill_between
            yerr_20v = np.sqrt((CCMD_20Vars_cov[ddp][sub]).diagonal())

            ax_main[i][j].plot(Mr_bins_x, CCMD_box_stat[ddp][sub], color='black', alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].plot(Mr_bins_x, CCMD_box_rsd_stat[ddp][sub], color='red', alpha=0.9, lw=3, ls=ls)

            ax_main[i][j].plot(Mr_bins_x, CCMD_20Vars_stat[ddp][sub], color='royalblue', alpha=0.9, lw=3, ls=ls)
            ax_main[i][j].fill_between(Mr_bins_x, CCMD_20Vars_stat[ddp][sub] - yerr_20v, 
                                       CCMD_20Vars_stat[ddp][sub] + yerr_20v, color='royalblue', alpha=0.2)
            
            # ================= 修改：比值图数据计算与绘制 (改为 bot) =================
            # 基准：Stitched CCMD Mean (c_mean)
            # 比值图仅展示 'bot' 部分：
            # 在主图顶部 ax_main[0][j] (Red) 和 ax_main[1][j] (Blue) 下方的比值图中
            # 基准Denominator始终是该 color_name 下的拼接后的 stitched_bot_c_mean

            # 1. 临时读取拼接后的 bot 数据作为 Denominator 基准
            c4_mean_bot  = CCMD_4Quads_stat[ddp][f'bot_{c_name}']
            c32_mean_bot = CCMD_stat[ddp][f'bot_{c_name}']
            stitched_bot_c_mean = np.concatenate([c4_mean_bot[:split_idx], c32_mean_bot[split_idx:]])
            denominator = stitched_bot_c_mean

            # 临时读取拼接后的 CCMD Bot Cov 对角std 作为 fill_between 误差范围底
            c4_cov_bot  = CCMD_4Quads_cov[ddp][f'bot_{c_name}']
            c32_cov_bot = CCMD_cov[ddp][f'bot_{c_name}']
            std_c4_bot  = np.sqrt(c4_cov_bot.diagonal())[:split_idx]
            std_c32_bot = np.sqrt(c32_cov_bot.diagonal())[split_idx:]
            stitched_bot_c_yerr = np.concatenate([std_c4_bot, std_c32_bot])

            # 2. 绘制 CCMD Cov 基准误差填充 (Forestgreen)
            # 以 1.0 为基准线，填充归一化后的 stitched_bot_c_yerr
            ax_ratio[i][j].fill_between(Mr_bins_x, 1.0 - (stitched_bot_c_yerr / denominator), 
                                        1.0 + (stitched_bot_c_yerr / denominator), color='forestgreen', alpha=0.3)
            ax_ratio[i][j].axhline(y=1.0, color='gray', linestyle='--', alpha=0.6) # 基准线

            # 3. 绘制 Numerator Ratios (solid lines only)
            # 均值部分：Box (black), Box RSD (red), 20Vars (royalblue)
            # 注意索引全改为了 f'bot_{c_name}'
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_box_stat[ddp][f'bot_{c_name}'] / denominator, color='black', alpha=0.9, lw=3, ls='--')
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_box_rsd_stat[ddp][f'bot_{c_name}'] / denominator, color='red', alpha=0.9, lw=3, ls='--')
            ax_ratio[i][j].plot(Mr_bins_x, CCMD_20Vars_stat[ddp][f'bot_{c_name}'] / denominator, color='royalblue', alpha=0.9, lw=3, ls='--')
            # ================================================================

# ==========================================
# 4. 图例与全局样式 (含刻度修复、灰色竖线、相对位置文本)
# ==========================================
# 提取一个样本数据用于生成图例底板 (借用 dummy_y 确保不在显示范围)
dummy_y = CCMD_stat['DDP18']['bot_red'] - 1 

ax_main[0][0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='-', label=r'$N_g^8$ top20%')
ax_main[0][0].plot(Mr_bins_x, dummy_y, color='k', alpha=0.9, lw=3, ls='--', label=r'$N_g^8$ bot20%')

ax_main[1][0].plot(Mr_bins_x, dummy_y, color='forestgreen', alpha=0.9, lw=3, ls='-', label='CCMD')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='black', alpha=0.9, lw=3, ls='-', label='CCMD box w/o rsd')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='red', alpha=0.9, lw=3, ls='-', label='CCMD box w/ rsd')
ax_main[1][0].plot(Mr_bins_x, dummy_y, color='royalblue', alpha=0.9, lw=3, ls='-', label='CCMD 20Vars box w/ rsd')

legend_0 = ax_main[0][0].legend(fontsize=22, loc='lower left', frameon=False)
legend_1 = ax_main[1][0].legend(fontsize=14, loc='lower left', frameon=False)
legend_1.get_texts()[0].set_color('forestgreen')
legend_1.get_texts()[2].set_color('red')
legend_1.get_texts()[3].set_color('royalblue')

# ================= 新增：在 ax_main[0][0] 添加相对位置文本 =================
ax_main[0][0].text(0.3, 0.82, 'overdense', transform=ax_main[0][0].transAxes,fontsize=16, color='gray', ha='center', va='center')
ax_main[0][0].text(0.3, 0.58, 'underdense', transform=ax_main[0][0].transAxes, fontsize=16, color='gray', ha='center', va='center')
# =========================================================================

# 全局样式循环设置
for i in range(2):
    for j in range(3):
        # 边框粗细统一设为2
        for spine in ax_main[i][j].spines.values(): spine.set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values(): spine.set_linewidth(2)

        # 主图刻度 (向内)
        ax_main[i][j].set(xlim=(-22.5, -17.5), ylim=(10**-5.4, 10**-2.2), yscale='log')
        ax_main[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)
        ax_main[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))

        # ================= 修改新增：比值图刻度与范围设置 =================
        ax_ratio[i][j].set(xlim=(-22.5, -17.5), ylim=(0.4, 1.6), yticks=[0.8, 1.2], yticklabels=['-20%', '20%']) 
        # 添加比值图文本 'Diff.' (gray, transform relative position)
        ax_ratio[i][j].text(0.95, 0.85, 'Diff.', transform=ax_ratio[i][j].transAxes, 
                             fontsize=16, color='gray', ha='right', va='center')

        ax_ratio[i][j].tick_params(axis='both', which='both', direction='in', top=True, right=True)
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_ratio[i][j].xaxis.set_minor_locator(ticker.MultipleLocator(0.4))
        # ================================================================

        # ================= 新增：在整数位置添加灰色竖线 =================
        for x_val in [-18, -19, -20, -21, -22]:
            ax_main[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5)
            ax_ratio[i][j].axvline(x=x_val, color='gray', alpha=0.4, linestyle='--', lw=1.5) # 比值图也加灰色竖线
        # ================================================================

        # 隐藏多余标签
        if j > 0: ax_main[i][j].set_yticklabels([]); ax_ratio[i][j].set_yticklabels([])
        if i == 0: ax_ratio[i][j].set_xticklabels([])

        # 反转X轴 (放在设置了刻度范围之后)
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

# 文本与标题
for j in range(3):
    ax_main[0][j].text(-21.4, 1.0e-3, 'Red', color='red', fontsize=24)
    ax_main[1][j].text(-21.3, 1.5e-3, 'Blue', color='royalblue', fontsize=24)

titles = [r'DDP: $-22 < M_r < -18$', r'DDP: $-22 < M_r < -19$', r'DDP: $-22 < M_r < -20$']
for j in range(3):
    ax_main[0][j].text(-17.9, 2.5e-3, titles[j], color='k', fontsize=22)


# ======================================================

fig.supylabel(r'$\tilde{\Phi}_{N_g^8}~[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=30, fontweight='bold', x=0.02)
fig.supxlabel(r'$M_r$', fontsize=30, fontweight='bold', y=0.02)

plt.tight_layout()
plt.show()

In [ ]:
### top diff 32Quads